# 1. Ingesta y curaduría

**Objetivo:** transformar la fuente experimental preliminar en una tabla trazable sin modificar el archivo original. Esta etapa registra procedencia, extrae la tabla principal, verifica el esquema y excluye únicamente los casos sin target. No imputa, escala ni entrena modelos.

**Entrada:** data/raw/dati_campagna_venditti.m.  
**Salidas:** versión intermedia, tabla procesada preliminar, manifiesto y reporte de calidad.

In [1]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src' / 'ingesta.py').is_file()
             and (p / 'config' / 'data_schema.yml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Abra Jupyter desde el repositorio o notebooks/.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import pandas as pd
import yaml
from src.ingesta import ingest
from src.preprocesamiento import curate_file

RAW_FILE = Path('data/raw/dati_campagna_venditti.m')
INTERIM_FILE = None
RUN_INGESTION = True
RUN_CURATION = True
schema = yaml.safe_load((ROOT / 'config' / 'data_schema.yml').read_text(encoding='utf-8'))
schema


{'project': {'task': 'regression',
  'target_concept': 'carga_primera_falla',
  'target_unit': 'N'},
 'columns': {'target': 'First_Failure_Load_N',
  'identifiers': ['Case_n'],
  'predictors': ['n_cells_Y',
   'Fiber_height',
   'Fiber_base',
   'Fiber_total_length',
   'Fiber_total_volume',
   'Pivot_height',
   'Pivot_radius',
   'Pivot_total_number',
   'Pivot_total_volume',
   'Sample_total_volume'],
  'groups': []},
 'predictor_units': {'n_cells_Y': 'conteo',
  'Fiber_height': 'mm',
  'Fiber_base': 'mm',
  'Fiber_total_length': 'mm',
  'Fiber_total_volume': 'mm^3',
  'Pivot_height': 'mm',
  'Pivot_radius': 'mm',
  'Pivot_total_number': 'conteo',
  'Pivot_total_volume': 'mm^3',
  'Sample_total_volume': 'mm^3'},
 'candidate_concepts': ['numero_celdas_direccion_y',
  'altura_fibra',
  'base_fibra',
  'altura_pivote',
  'radio_pivote',
  'longitud_total_fibras',
  'volumen_total_fibras',
  'numero_total_pivotes',
  'volumen_total_pivotes',
  'volumen_total_especimen'],
 'curation': {'

## Controles aplicados

- identificación de la fuente mediante SHA-256;
- extracción segura de la tabla samples, sin ejecutar MATLAB;
- validación de columnas, tipos, unidades y roles;
- registro de faltantes, duplicados, constantes y exclusiones;
- separación entre variables geométricas y respuestas posensayo.

La salida se denomina **preliminar** porque todavía puede ampliarse y requiere confirmar réplicas, grupos y el criterio experimental exacto de primera falla.

In [2]:
if RUN_INGESTION:
    if RAW_FILE is None:
        raise ValueError('Indique un archivo original real.')
    _, manifest = ingest(Path(RAW_FILE))
    INTERIM_FILE = manifest['interim_file']
    display(manifest)
else:
    print('Ingesta preparada; no se ha ejecutado.')


{'manifest_version': 1,
 'run_id': '20261003T015928118917Z',
 'source_file': 'data/raw/dati_campagna_venditti.m',
 'sha256': '3d82f876bb8ec652184707235aa4df1dfb4b1a6615551f78fe0e0662a6723f0c',
 'source_modified_at_utc': '2026-07-07T03:10:09.424381+00:00',
 'ingested_at_utc': '2026-10-03T01:59:28.128495+00:00',
 'rows': 27,
 'columns': 20,
 'column_names': ['Case_n',
  'n_cells_Y',
  'Fiber_height',
  'Fiber_base',
  'Fiber_total_length',
  'Fiber_total_volume',
  'Pivot_height',
  'Pivot_radius',
  'Pivot_total_number',
  'Pivot_total_volume',
  'Sample_total_volume',
  'First_Failure_Load_N',
  'Ultimate_Load',
  'Maximum_Displacement',
  'residual_Displacement_1cycle_10mm',
  'residual_Displacement_2cycle_20mm',
  'residual_Displacement_3cycle_30mm',
  'residual_Displacement_4cycle_40mm',
  'residual_Displacement_5cycle_50mm',
  'residual_Displacement_6cycle_60mm'],
 'reader': {'format': '.m',
  'parser': 'src.importar_matlab.load_matlab_samples',
  'extraction': 'tabla samples; asig

In [3]:
quality = None
if RUN_CURATION:
    if INTERIM_FILE is None:
        raise ValueError('Indique una ingesta real versionada.')
    quality = curate_file(Path(INTERIM_FILE), ROOT / 'config' / 'data_schema.yml')
    display(pd.DataFrame(quality['excluded_records']))
    display(pd.DataFrame({'antes': quality['before']['missing_by_column'],
                          'después': quality['after']['missing_by_column']}))
    print('Tabla:', quality['output_file'])
else:
    print('Curaduría preparada; requiere completar y verificar el esquema.')


,source_record,reason
0,2,missing_target


,antes,después
Case_n,0,0.0
n_cells_Y,0,0.0
Fiber_height,0,0.0
Fiber_base,0,0.0
Fiber_total_length,0,0.0
Fiber_total_volume,0,0.0
Pivot_height,0,0.0
Pivot_radius,0,0.0
Pivot_total_number,0,0.0
Pivot_total_volume,0,0.0


Tabla: data/processed/20261003T015928287717Z/model_table.csv


## Resultado de la etapa

La ejecución conserva tres niveles:

1. **Raw:** archivo original recibido, sin modificaciones.
2. **Interim:** extracción tabular automática y trazable; todavía contiene todas las respuestas de la fuente.
3. **Processed preliminar:** tabla analítica con identificador, predictores autorizados y target disponible.

La siguiente etapa es 02_EDA.ipynb. El modelado permanece separado hasta aprobar el protocolo de partición.